In [1]:
"""
Statistical Significance Testing for Stacking Ensemble Tc Prediction
=====================================================================
Tests performed:
  - Paired t-test
  - Wilcoxon signed-rank test

Comparison: Stacking Ensemble vs each individual base learner.
Metric used: absolute residuals per fold (|y_true - y_pred|),
             computed via 10-fold cross-validation on the training set.

This is the correct setup because:
  1. Paired tests require one error value per sample/fold — the pairing
     comes from evaluating both models on the exact same CV folds.
  2. Using the test set directly gives only one scalar per model,
     which is insufficient for a paired test. CV gives 10 paired values.
  3. Fold-level RMSE is used as the paired unit (one value per fold,
     10 folds -> 10 pairs), consistent with standard ML comparison practice.
"""

import warnings

from sklearn.linear_model import Ridge
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import joblib

from sklearn.preprocessing import RobustScaler
from sklearn.ensemble import StackingRegressor
from sklearn.model_selection import KFold, cross_val_predict
from sklearn.metrics import mean_squared_error
from scipy.stats import ttest_rel, wilcoxon
from xgboost import XGBRegressor

# =============================================================================
# 1. LOAD DATA
# =============================================================================
df_train = pd.read_csv("top30_vF_xi_Hc_train_83.csv")

TARGET  = "critical_temp"
X_train = df_train.drop(columns=[TARGET])
y_train = df_train[TARGET].values

scaler      = RobustScaler()
X_train_sc  = scaler.fit_transform(X_train)

print(f"Training set : {X_train.shape}")

# =============================================================================
# 2. LOAD BASE LEARNERS
# =============================================================================
MODEL_DIR = "models_83/tuning_50_HPO"

knn_best  = joblib.load(f"{MODEL_DIR}/knn_best.pkl")
svr_best  = joblib.load(f"{MODEL_DIR}/svr_best.pkl")
mlp_best  = joblib.load(f"{MODEL_DIR}/mlp_best.pkl")
best_rf   = joblib.load(f"{MODEL_DIR}/best_rf.pkl")
best_lgb  = joblib.load(f"{MODEL_DIR}/best_lgb.pkl")

base_models = {
    "KNN"          : knn_best,
    "SVR"          : svr_best,
    "MLP"          : mlp_best,
    "Random Forest": best_rf,
    "LightGBM"     : best_lgb,
}

# meta_xgb = XGBRegressor(
#     n_estimators     = 300,
#     learning_rate    = 0.05,
#     max_depth        = 4,
#     colsample_bytree = 0.6,
#     subsample        = 0.6,
#     min_child_weight = 5,
#     gamma            = 0.1,
#     reg_lambda       = 10.0,
#     reg_alpha        = 0.5,
#     random_state     = 42,
#     verbosity        = 0,
#     n_jobs           = -1,
# )

# stack = StackingRegressor(
#     estimators      = list(base_models.items()),
#     final_estimator = meta_xgb,
#     cv              = KFold(n_splits=5, shuffle=True, random_state=42),
#     passthrough     = True,
#     n_jobs          = -1,
# )


stack = StackingRegressor(
    estimators     = list(base_models.items()),
    final_estimator= Ridge(alpha=2.0, random_state=42),
    cv             = KFold(n_splits=5, shuffle=True, random_state=42),
    passthrough    = True,
    n_jobs         = -1,
)

# =============================================================================
# 3. 10-FOLD CV TO GENERATE FOLD-LEVEL RMSE FOR EACH MODEL
#    The paired test unit is per-fold RMSE, not per-sample residual.
#    This gives 10 paired observations per comparison — standard practice.
# =============================================================================
CV = KFold(n_splits=5, shuffle=True, random_state=42)

def fold_rmse_scores(model, X, y, cv):
    """
    Return a list of RMSE values, one per CV fold.
    Uses cross_val_predict to get OOF predictions, then
    splits them back into folds to get fold-level RMSE.
    """
    fold_rmses = []
    for train_idx, val_idx in cv.split(X):
        model.fit(X[train_idx], y[train_idx])
        preds = model.predict(X[val_idx])
        rmse  = np.sqrt(mean_squared_error(y[val_idx], preds))
        fold_rmses.append(rmse)
    return np.array(fold_rmses)

print("\nComputing 10-fold CV RMSE for each model...")
print("(This takes a few minutes for the stacking ensemble)\n")

fold_scores = {}

for name, model in base_models.items():
    scores = fold_rmse_scores(model, X_train_sc, y_train, CV)
    fold_scores[name] = scores
    print(f"  {name:<18}: mean RMSE = {scores.mean():.4f} K  "
          f"(std = {scores.std():.4f} K)")

fold_scores["Stacking"] = fold_rmse_scores(stack, X_train_sc, y_train, CV)
print(f"  {'Stacking':<18}: mean RMSE = {fold_scores['Stacking'].mean():.4f} K  "
      f"(std = {fold_scores['Stacking'].std():.4f} K)")

# =============================================================================
# 4. PAIRED T-TEST AND WILCOXON SIGNED-RANK TEST
#    H0: mean fold-RMSE of stacking == mean fold-RMSE of base model
#    H1: mean fold-RMSE of stacking <  mean fold-RMSE of base model
#    (one-tailed: testing stacking is better, i.e., lower RMSE)
# =============================================================================
ALPHA = 0.05

print("\n" + "=" * 78)
print(f"{'Comparison':<32}{'t-stat':>9}{'t p-val':>12}"
      f"{'W-stat':>10}{'W p-val':>12}{'Sig?':>8}")
print("-" * 78)

results_records = []

stack_scores = fold_scores["Stacking"]

for name in base_models.keys():
    base_scores = fold_scores[name]
    diff        = base_scores - stack_scores   # positive = stacking better

    # paired t-test (one-tailed: stacking < base)
    t_stat, t_p_two = ttest_rel(base_scores, stack_scores)
    t_p_one = t_p_two / 2 if t_stat > 0 else 1.0 - t_p_two / 2

    # Wilcoxon signed-rank test (one-tailed)
    try:
        w_stat, w_p_two = wilcoxon(base_scores, stack_scores,
                                    alternative="greater")
        w_p_one = w_p_two
    except ValueError:
        # identical distributions (all differences zero)
        w_stat, w_p_one = np.nan, 1.0

    sig = "YES" if (t_p_one < ALPHA and w_p_one < ALPHA) else "NO"

    label = f"Stacking vs {name}"
    print(f"  {label:<30}{t_stat:>9.4f}{t_p_one:>12.4f}"
          f"{w_stat:>10.2f}{w_p_one:>12.4f}{sig:>8}")

    results_records.append({
        "Comparison"    : label,
        "t_statistic"   : round(t_stat,  4),
        "t_test_p_value": round(t_p_one, 4),
        "W_statistic"   : round(float(w_stat), 2) if not np.isnan(w_stat) else np.nan,
        "Wilcoxon_p_value": round(w_p_one, 4),
        "Significant"   : sig,
        "Stack_mean_RMSE": round(stack_scores.mean(), 4),
        "Base_mean_RMSE" : round(base_scores.mean(),  4),
    })

print("=" * 78)
print(f"Significance threshold alpha = {ALPHA}  (one-tailed)")
print("'Sig?' = YES if BOTH t-test AND Wilcoxon p-value < alpha\n")

# =============================================================================
# 5. SAVE RESULTS TABLE
# =============================================================================
df_stats = pd.DataFrame(results_records)
df_stats.to_csv("statistical_test_results.csv", index=False)
print("Saved table -> statistical_test_results.csv")

# =============================================================================
# 6. PLOTS
# =============================================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

PALETTE = {
    "KNN"          : "#e41a1c",
    "SVR"          : "#377eb8",
    "MLP"          : "#4daf4a",
    "Random Forest": "#984ea3",
    "LightGBM"     : "#ff7f00",
    "Stacking"     : "#000000",
}

# --- (a) Box plot of fold-RMSE per model ---
ax = axes[0]
all_names  = list(base_models.keys()) + ["Stacking"]
box_data   = [fold_scores[n] for n in all_names]
box_colors = [PALETTE[n] for n in all_names]

bp = ax.boxplot(box_data, patch_artist=True, notch=False,
                medianprops=dict(color="white", linewidth=2))
for patch, color in zip(bp["boxes"], box_colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.75)

ax.set_xticks(range(1, len(all_names) + 1))
ax.set_xticklabels(all_names, rotation=20, ha="right", fontsize=9)
ax.set_ylabel("Fold RMSE (K)", fontsize=11)
ax.set_title("(a) 10-Fold CV RMSE Distribution per Model", fontsize=11)
ax.grid(True, alpha=0.3, axis="y")

# highlight stacking box
bp["boxes"][-1].set_linewidth(2.0)
bp["boxes"][-1].set_edgecolor("black")

# --- (b) p-value comparison bar chart ---
ax2 = axes[1]
comparisons = [r["Comparison"].replace("Stacking vs ", "") for r in results_records]
t_pvals     = [r["t_test_p_value"] for r in results_records]
w_pvals     = [r["Wilcoxon_p_value"] for r in results_records]

x     = np.arange(len(comparisons))
width = 0.35

bars_t = ax2.bar(x - width/2, t_pvals, width, label="Paired t-test",
                  color="#377eb8", alpha=0.8, edgecolor="white")
bars_w = ax2.bar(x + width/2, w_pvals, width, label="Wilcoxon signed-rank",
                  color="#e41a1c", alpha=0.8, edgecolor="white")

ax2.axhline(ALPHA, color="black", lw=1.2, ls="--",
             label=f"α = {ALPHA}")
ax2.set_xticks(x)
ax2.set_xticklabels(comparisons, rotation=20, ha="right", fontsize=9)
ax2.set_ylabel("p-value (one-tailed)", fontsize=11)
ax2.set_title("(b) Statistical Test p-Values\n"
               "(Stacking vs each base learner)", fontsize=11)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3, axis="y")

# annotate significance
for i, rec in enumerate(results_records):
    y_pos = max(rec["t_test_p_value"], rec["Wilcoxon_p_value"]) + 0.005
    label = "*" if rec["Significant"] == "YES" else "ns"
    ax2.text(i, y_pos, label, ha="center", fontsize=11,
             color="green" if label == "*" else "gray")

plt.tight_layout()
plt.savefig("statistical_tests.png", dpi=150,
            bbox_inches="tight", facecolor="white")
plt.close()
print("Saved plot  -> statistical_tests.png")
print("\nDone.")

Training set : (17010, 30)

Computing 10-fold CV RMSE for each model...
(This takes a few minutes for the stacking ensemble)

  KNN               : mean RMSE = 9.5694 K  (std = 0.2929 K)
  SVR               : mean RMSE = 9.1647 K  (std = 0.3511 K)
  MLP               : mean RMSE = 9.1557 K  (std = 0.2546 K)
  Random Forest     : mean RMSE = 9.2910 K  (std = 0.3687 K)
  LightGBM          : mean RMSE = 9.1494 K  (std = 0.2982 K)
  Stacking          : mean RMSE = 8.9664 K  (std = 0.3100 K)

Comparison                         t-stat     t p-val    W-stat     W p-val    Sig?
------------------------------------------------------------------------------
  Stacking vs KNN                 13.5287      0.0001     15.00      0.0312     YES
  Stacking vs SVR                  9.3040      0.0004     15.00      0.0312     YES
  Stacking vs MLP                  5.0772      0.0035     15.00      0.0312     YES
  Stacking vs Random Forest        6.6062      0.0014     15.00      0.0312     YES
  Stacki